In [1]:
import numpy as np

# Constants
R = 8.314462618  # J/(mol·K), ideal gas constant
P_ATM = 101325.0  # Pa (1 atm)
M_H2O = 18.01528e-3  # kg/mol
M_DRY_AIR = 28.96546e-3  # kg/mol

# Atomic weights (g/mol)
atomic_weights = {
    'C': 12.011,
    'N': 14.007,
    'O': 15.999,
    'Ar': 39.948,
    'H': 1.008
}

# G4_AIR dry air atomic fractions
dry_air_atomic_fractions = {
    'C': 0.000124,
    'N': 0.755268,
    'O': 0.231781,
    'Ar': 0.012827
}

def saturation_vapor_pressure(T_K):
    """Returns saturation vapor pressure of water in Pa using Tetens formula."""
    T_C = T_K - 273.15
    return 6.1078 * 10**((7.5 * T_C) / (T_C + 237.3)) * 100  # in Pa

def humid_air_properties(T_K=298.15, RH=0.5, P=P_ATM):
    # Compute vapor pressure
    p_sat = saturation_vapor_pressure(T_K)
    p_H2O = RH * p_sat
    p_dry = P - p_H2O

    # Mole fraction of water vapor
    y_H2O = p_H2O / P

    # Effective molar mass of humid air
    M_humid = y_H2O * M_H2O + (1 - y_H2O) * M_DRY_AIR  # kg/mol

    # Density of humid air using ideal gas law
    rho = (P * M_humid) / (R * T_K)  # kg/m³
    rho_gcm3 = rho * 1e-3  # convert to g/cm³

    # Atomic fractions
    total_atoms = {}
    for el, frac in dry_air_atomic_fractions.items():
        total_atoms[el] = frac * (1 - y_H2O)

    # Add H2O contributions: H: 2 atoms, O: 1 atom per molecule
    total_atoms['H'] = 2 / 3 * y_H2O
    total_atoms['O'] = total_atoms.get('O', 0.0) + 1 / 3 * y_H2O

    # Normalize atomic fractions
    total = sum(total_atoms.values())
    for el in total_atoms:
        total_atoms[el] /= total

    return {
        'temperature_K': T_K,
        'relative_humidity': RH,
        'pressure_Pa': P,
        'density_g_per_cm3': rho_gcm3,
        'molar_mass_kg_per_mol': M_humid,
        'atomic_fractions': total_atoms
    }


In [7]:

# Example usage
result = humid_air_properties(T_K=298.15, RH=0.25)  # 25°C, 50% RH
print("Density (g/cm³):", result['density_g_per_cm3'])
print("Molar mass (kg/mol):", result['molar_mass_kg_per_mol'])
print("Atomic Fractions:")
for element, fraction in result['atomic_fractions'].items():
    print(f"  {element}: {fraction:.6f}")

Density (g/cm³): 0.0011804376014398646
Molar mass (kg/mol): 0.028879882457857414
Atomic Fractions:
  C: 0.000123
  N: 0.749365
  O: 0.232575
  Ar: 0.012727
  H: 0.005210
